# EIFICA Result Visualization

## 0. Style and constants

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, LogLocator, NullFormatter
import seaborn as sns
%matplotlib inline

plt.style.use("seaborn-v0_8-whitegrid")
mpl.rcParams.update({
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
    "figure.dpi": 120, "savefig.dpi": 300,
    "axes.linewidth": 1.1, "axes.edgecolor": "#333333",
    "axes.titlepad": 10, "axes.labelpad": 6,
    "grid.color": "#c9c9c9", "grid.linewidth": 0.7, "grid.alpha": 0.6,
    "legend.frameon": True, "legend.framealpha": 0.92,
    "legend.edgecolor": "#cccccc", "legend.fancybox": True,
    "xtick.direction": "out", "ytick.direction": "out",
})

EPS_THETA = [(0.08, 0.12), (0.05, 0.10), (0.10, 0.15), (0.03, 0.06)]
PANEL     = ["(a)", "(b)", "(c)", "(d)"]
NETS   = ["case24_ieee_rts", "case118"]
NETLAB = {"case24_ieee_rts": "IEEE RTS-24", "case118": "IEEE-118"}

MCLR = {"CVaR": "#B23A48", "FICA": "#2F4B7C",
        "EIFICA-1shot": "#E08E0B", "EIFICA-iter": "#2E8B57"}
MMK  = {"CVaR": "o", "FICA": "D", "EIFICA-1shot": "s", "EIFICA-iter": "^"}
ACCENT = "#C00000"
LABELS = {"CVaR": "CVaR", "FICA": "FICA",
          "EIFICA-1shot": "EIFICA (direct)", "EIFICA-iter": "EIFICA (iterative)"}
PREF = ["CVaR", "FICA", "EIFICA-1shot", "EIFICA-iter"]

NB_DIR = os.getcwd()
ROOT   = os.path.abspath(os.path.join(NB_DIR, ".."))
FIGDIR = os.path.join(NB_DIR, "figures"); os.makedirs(FIGDIR, exist_ok=True)

## 1. Load data -> long table `res`

In [ ]:
f = sorted(glob.glob(os.path.join(ROOT, "master_summary_*.csv")))[-1]
d = pd.read_csv(f)
for c in ["min_cost","reliability_pct","t_solve_s","peak_rss_mb","epsilon","theta","N_WDR","seed"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d["grp"] = np.where(d.source_dir.str.contains("speed"), "speed", "exact")

def _relabel(r):
    if r.method == "CVAR": return "CVaR"
    if r.method == "FICA": return "FICA"
    return "EIFICA-1shot" if r.grp == "speed" else "EIFICA-iter"
d["m"] = d.apply(_relabel, axis=1)

res = d.rename(columns={"epsilon":"eps","N_WDR":"N","t_solve_s":"Time",
                        "reliability_pct":"JCC","min_cost":"Cost","peak_rss_mb":"Mem"})[
    ["network","eps","theta","N","seed","m","Time","JCC","Cost","Mem"]]

ORDER = [m for m in PREF if m in set(res.m.unique())]
print("long table rows:", len(res), "| methods:", ORDER, "| source:", os.path.basename(f))
res.head()

## 2. Generic faceted plotting function

In [ ]:
def _style_axes(ax):
    """Unify axis styling: hide top/right spines, grid below, refine ticks."""
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color("#333333")
    ax.set_axisbelow(True)
    ax.grid(True, which="major", linestyle="--", linewidth=1.1, alpha=0.6)
    ax.grid(True, which="minor", linestyle=":",  linewidth=0.7, alpha=0.3)
    ax.tick_params(axis="both", which="major", labelsize=16, length=4, width=1.0)
    for lb in ax.get_xticklabels() + ax.get_yticklabels():
        lb.set_fontweight("bold")

def facet(net, ycol, ylab, logy=False, target=False, annotate_speedup=False,
          yconv=1.0, fname=None, legend_loc="best", stat="median", shade=True,
          zoom_inset=False, shared_legend=False):
    """2x2 facets (per eps,theta): center line (stat=median/mean) + optional IQR band (shade) + hollow markers + speedup annotation.
    stat sets the statistic for both the center line and the speedup; shade=False draws no band."""
    g = res[res.network == net]
    fig, axs = plt.subplots(2, 2, figsize=(10, 7.1), sharex=False)
    axs = axs.flatten()
    for i, (eps, th) in enumerate(EPS_THETA):
        ax = axs[i]
        sub = g[(g.eps == eps) & (g.theta == th)].copy()
        sub[ycol] = sub[ycol] * yconv
        so = [k for k in ORDER if k in set(sub.m.unique())]
        Ns = np.sort(sub.N.unique())
        axins = ax.inset_axes([0.36, 0.26, 0.42, 0.42]) if zoom_inset else None
        for k in so:
            gk = sub[sub.m == k].groupby("N")[ycol]
            center = gk.agg(stat).reindex(Ns)
            col = MCLR[k]
            if shade:
                lo, hi = gk.quantile(0.25).reindex(Ns), gk.quantile(0.75).reindex(Ns)
                ax.fill_between(Ns, lo, hi, color=col, alpha=0.13, linewidth=0, zorder=1.5)
                for _b in (lo, hi):
                    ax.plot(Ns, _b, color=col, lw=0.7, ls=(0, (3, 2)), alpha=0.75, zorder=2)
            ax.plot(Ns, center, color=col, linewidth=2.4, marker=MMK[k], markersize=8,
                    markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.8,
                    zorder=3, solid_capstyle="round")
            if axins is not None:
                if shade:
                    axins.fill_between(Ns, lo, hi, color=col, alpha=0.13, linewidth=0, zorder=1.5)
                    for _b in (lo, hi):
                        axins.plot(Ns, _b, color=col, lw=0.7, ls=(0, (3, 2)), alpha=0.75, zorder=2)
                axins.plot(Ns, center, color=col, linewidth=2.0, marker=MMK[k], markersize=6,
                           markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5, zorder=3)
        ax.set_title(rf"{PANEL[i]}  $\boldsymbol{{\epsilon}}$={eps},  $\boldsymbol{{\theta}}$={th}",
                     fontsize=20, fontweight="bold", loc="left", color="#222222")
        ax.set_xlabel(r"$\boldsymbol{N}$ (Training Scenarios)", fontsize=16, fontweight="bold")
        ax.set_ylabel(ylab if i % 2 == 0 else "", fontsize=16, fontweight="bold")
        ax.set_xticks(Ns)
        if logy:
            ax.set_yscale("log")
            ax.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0, 2.0, 5.0), numticks=12))
            ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=(3.0, 4.0, 6.0, 7.0, 8.0, 9.0), numticks=12))
            ax.yaxis.set_minor_formatter(NullFormatter())
            _sf = ScalarFormatter(); _sf.set_scientific(False)
            ax.yaxis.set_major_formatter(_sf)
        _style_axes(ax)
        if target:
            ax.axhline((1 - eps) * 100, color="#c0392b", linestyle=(0, (4, 3)),
                       linewidth=1.7, zorder=1)
        if axins is not None:
            zmask = (Ns >= 80) & (Ns <= 100)
            zN = Ns[zmask]
            vals = []
            for k in so:
                gk2 = sub[sub.m == k].groupby("N")[ycol]
                series = [gk2.quantile(0.25), gk2.quantile(0.75)] if shade else [gk2.agg(stat)]
                for s in series:
                    vals += list(pd.Series(s.reindex(Ns)[zmask]).dropna())
            _zlo, _zhi = min(vals), max(vals); _zp = (_zhi - _zlo) * 0.18 + 0.05
            _tgt = (1 - eps) * 100
            if target and _zlo - _zp <= _tgt <= _zhi + _zp:
                axins.axhline(_tgt, color="#c0392b", linestyle=(0, (4, 3)), linewidth=1.4, zorder=1)
            axins.set_xlim(zN.min() - 4, zN.max() + 4)
            axins.set_ylim(_zlo - _zp, _zhi + _zp)
            axins.set_xticks(zN)
            axins.tick_params(labelsize=11)
            for _lb in axins.get_xticklabels() + axins.get_yticklabels():
                _lb.set_fontweight("bold")
            axins.grid(True, linestyle="--", linewidth=0.8, alpha=0.5)
            for _sp in axins.spines.values():
                _sp.set_edgecolor("#888888")
            ax.indicate_inset_zoom(axins, edgecolor="#888888", alpha=0.8, linewidth=1.2)
        if annotate_speedup and {"FICA", "EIFICA-1shot"} <= set(so):
            tf = sub[sub.m == "FICA"].groupby("N")[ycol].agg(stat)
            to = sub[sub.m == "EIFICA-1shot"].groupby("N")[ycol].agg(stat)
            if logy:
                ax.set_ylim(top=ax.get_ylim()[1] * 1.9)
            for n in tf.index.intersection(to.index):
                if n >= 100 and to[n] > 0 and tf[n] / to[n] >= 1.5:
                    ax.annotate(f"{tf[n]/to[n]:.1f}×", xy=(n, tf[n]), xytext=(0, 8),
                                textcoords="offset points", ha="center", va="bottom",
                                fontsize=15, fontweight="bold", color=ACCENT,
                                fontname="STIXGeneral", zorder=6, clip_on=False)
        lines  = [Line2D([0],[0], color=MCLR[k], marker=MMK[k], linestyle="-",
                         markerfacecolor="white", markeredgecolor=MCLR[k],
                         markeredgewidth=1.6, markersize=8, linewidth=2.2) for k in so]
        labels = [LABELS[k] for k in so]
        if target:
            lines.append(Line2D([0],[0], color="#c0392b", linestyle=(0,(4,3)), linewidth=1.7))
            labels.append(r"target $1-\epsilon$")
        if shared_legend:
            _shared = (lines, labels)
        else:
            ax.legend(lines, labels, loc=legend_loc, prop={"weight":"bold","size":12})
    if shared_legend:
        _lh, _ll = _shared
        fig.tight_layout(rect=(0, 0, 1, 0.93))
        _p0 = axs[0].get_position(); _p1 = axs[1].get_position()
        fig.legend(_lh, _ll, bbox_to_anchor=(_p0.x0, 0.94, _p1.x1 - _p0.x0, 0.05),
                   loc="lower left", ncol=len(_ll), mode="expand", borderaxespad=0.0,
                   prop={"weight":"bold","size":15}, frameon=True, framealpha=0.95,
                   edgecolor="#cccccc", handletextpad=0.5)
    else:
        fig.tight_layout()
    if fname:
        for ext in ("pdf","png"):
            fig.savefig(os.path.join(FIGDIR, f"{fname}.{ext}"), bbox_inches="tight")
    plt.show()

## 3. Solve time vs N (log axis, direct speedup annotated)

In [ ]:
facet("case24_ieee_rts", "Time", "Solve Time (s)", logy=True, annotate_speedup=True, fname="time_c24", shared_legend=True)

In [ ]:
facet("case118", "Time", "Solve Time (s)", logy=True, annotate_speedup=True, fname="time_c118", shared_legend=True)

### 3b. Supplement: no shading + mean speedup

In [ ]:
facet("case24_ieee_rts", "Time", "Solve Time (s)", logy=True, annotate_speedup=True, fname="time_c24_mean", stat="mean", shade=False, shared_legend=True)

In [ ]:
facet("case118", "Time", "Solve Time (s)", logy=True, annotate_speedup=True, fname="time_c118_mean", stat="mean", shade=False, shared_legend=True)

## 4. Out-of-sample reliability vs N (red dashed = target $1-\epsilon$)

In [ ]:
facet("case24_ieee_rts", "JCC", "JCC Satisfaction (%)", target=True, fname="reliability_c24", zoom_inset=True, shared_legend=True)

In [ ]:
facet("case118", "JCC", "JCC Satisfaction (%)", target=True, fname="reliability_c118", zoom_inset=True, shared_legend=True)

## 5. Peak memory vs N (log axis, GB)

In [ ]:
facet("case24_ieee_rts", "Mem", "Peak Memory (GB)", logy=True, yconv=1/1024.0, fname="memory_c24", shared_legend=True)

In [ ]:
facet("case118", "Mem", "Peak Memory (GB)", logy=True, yconv=1/1024.0, fname="memory_c118", shared_legend=True)

## 6. Summary table (conservative ratio-of-means)

In [ ]:
rows = []
for net in NETS:
    g = res[res.network == net]
    for N, gg in g.groupby("N"):
        tC = gg[gg.m=="CVaR"].Time.mean()
        n_cvar_ok = gg[(gg.m=="CVaR") & gg.Time.notna()].shape[0]
        tF, tO, tX = gg[gg.m=="FICA"].Time.mean(), gg[gg.m=="EIFICA-1shot"].Time.mean(), gg[gg.m=="EIFICA-iter"].Time.mean()
        mF, mO = gg[gg.m=="FICA"].Mem.mean(), gg[gg.m=="EIFICA-1shot"].Mem.mean()
        jF, jO = gg[gg.m=="FICA"].JCC.mean(), gg[gg.m=="EIFICA-1shot"].JCC.mean()
        f2 = gg[gg.m=="FICA"].set_index("seed").Cost; x2 = gg[gg.m=="EIFICA-iter"].set_index("seed").Cost
        rel = ((x2 - f2).abs()/f2.abs()).max()
        rows.append(dict(network=NETLAB[net], N=int(N),
                         CVaR_t=(round(tC,1) if pd.notna(tC) else np.nan), CVaR_ok=n_cvar_ok,
                         FICA_t=round(tF,1), one_t=round(tO,1), iter_t=round(tX,1),
                         speedup_one=round(tF/tO,2), speedup_iter=round(tF/tX,2),
                         mem_ratio=round(mF/mO,1), jcc_gap_one=round(jO-jF,3),
                         obj_reldiff_iter=f"{rel:.1e}"))
tbl = pd.DataFrame(rows)
tbl.to_csv(os.path.join(NB_DIR, "summary_table.csv"), index=False)

sty = (tbl.style
       .background_gradient(cmap="Greens", subset=["speedup_one","speedup_iter","mem_ratio"])
       .background_gradient(cmap="RdYlGn", subset=["jcc_gap_one"], vmin=-3, vmax=0)
       .format({"speedup_one":"{:.2f}×","speedup_iter":"{:.2f}×","mem_ratio":"{:.1f}×",
                "jcc_gap_one":"{:+.2f}","FICA_t":"{:.1f}","one_t":"{:.1f}","iter_t":"{:.1f}"})
       .set_caption("Ratio-of-means summary (green = larger gain; red = larger JCC drop)")
       .set_properties(**{"text-align":"center","font-size":"11pt"})
       .set_table_styles([{"selector":"caption","props":[("font-weight","bold"),("font-size","12pt"),("padding","6px")]},
                          {"selector":"th","props":[("text-align","center"),("background-color","#f2f2f2")]}]))
sty

## 7. Optimality Gap Analysis

In [ ]:
import matplotlib.ticker as ticker

_ogf = os.path.join(ROOT, "Solar_c24_optgap", "optimality_gap_T3_fixed.csv")
og = pd.read_csv(_ogf)
og = og.dropna(subset=["obj"]).drop_duplicates(subset=["method","sf","epsilon","n_wdr","seed"], keep="last")
piv = og.pivot_table(index=["sf","epsilon","n_wdr","seed"], columns="method", values="obj").reset_index()
piv["diff_pct"] = (piv["EIFICA"] - piv["ExactLHS"]) / piv["ExactLHS"] * 100

SF_list, EPS_list = sorted(piv.sf.unique()), sorted(piv.epsilon.unique())
PAL_N = ["#E68685", "#9986B1", "#F3983C", "#6DAB81"]
gmean, gmin, gmax = piv.diff_pct.mean(), piv.diff_pct.min(), piv.diff_pct.max()

fig, axes = plt.subplots(len(SF_list), len(EPS_list), figsize=(10, 3.7*len(SF_list)),
                         sharex=True, sharey=False)
axes = np.atleast_2d(axes)
for i, sf in enumerate(SF_list):
    for j, eps in enumerate(EPS_list):
        ax = axes[i, j]
        sub = piv[(piv.sf==sf) & (piv.epsilon==eps)]
        if not sub.empty:
            sns.boxplot(x="n_wdr", y="diff_pct", data=sub, ax=ax, hue="n_wdr",
                        palette=PAL_N, legend=False, width=0.6, showfliers=False)
            sns.stripplot(x="n_wdr", y="diff_pct", data=sub, ax=ax, jitter=True, size=4.5,
                          marker="o", facecolors="none", edgecolor="red", linewidth=0.8, alpha=0.8)
            pmin, pmax = sub.diff_pct.min(), sub.diff_pct.max()
            pad = max((pmax - pmin) * 0.35, 0.01)
            ax.set_ylim(pmin - pad, pmax + pad)
        ax.set_title(rf"$\boldsymbol{{SF}}$={sf}, $\boldsymbol{{\epsilon}}$={eps}", fontsize=20, fontweight="bold")
        ax.set_xlabel(r"$\boldsymbol{N}$ (Sample Size)" if i==len(SF_list)-1 else "", fontsize=16, fontweight="bold")
        ax.set_ylabel(r"Obj. Diff. (%)" if j==0 else "", fontsize=16, fontweight="bold")
        _style_axes(ax); ax.yaxis.set_major_locator(ticker.MaxNLocator(5))
fig.suptitle(rf"Optimality gap: EIFICA vs exact ExactLHS  (mean {gmean:.2f}%, range [{gmin:.2f}, {gmax:.2f}]%)  —  IEEE RTS-24, T=3",
             fontsize=14.5, fontweight="bold", y=0.998)
fig.tight_layout(rect=(0,0,1,0.99))
for ext in ("pdf","png"):
    fig.savefig(os.path.join(FIGDIR, f"optimality_gap.{ext}"), bbox_inches="tight")
plt.show()

_ntl = int((og.status==9).sum()) if "status" in og.columns else 0
print(f"pairs={len(piv)} | gap mean={gmean:.3f}% median={piv.diff_pct.median():.3f}% "
      f"range[{gmin:.3f}, {gmax:.3f}]% | all_positive={bool((piv.diff_pct>0).all())}")
if _ntl:
    print(f"note: {_ntl} ExactLHS hit the time limit (status=9), included (gap slightly underestimated).")

### 7b. Shared y-axis + zoomed inset

In [ ]:
def _dg(ax, sub, zoom=False):
    sns.boxplot(x="n_wdr", y="diff_pct", data=sub, ax=ax, hue="n_wdr", palette=PAL_N,
                legend=False, width=0.6, showfliers=False)
    sns.stripplot(x="n_wdr", y="diff_pct", data=sub, ax=ax, jitter=True,
                  size=(3.2 if zoom else 4.5), marker="o", facecolors="none",
                  edgecolor="red", linewidth=0.8, alpha=0.85)

_lo = min(0.0, gmin); _pad = (gmax - _lo) * 0.10
_rc = plt.rc_context({"font.weight": "bold"})
_rc.__enter__()
fig, axes = plt.subplots(len(SF_list), len(EPS_list), figsize=(10, 3.83*len(SF_list)),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for i, sf in enumerate(SF_list):
    for j, eps in enumerate(EPS_list):
        ax = axes[i, j]
        sub = piv[(piv.sf==sf) & (piv.epsilon==eps)]
        _dg(ax, sub)
        ax.axhline(0, color="#2F4B7C", linestyle="-", linewidth=2, zorder=1)
        ax.set_title(rf"$\boldsymbol{{SF}}$={sf}, $\boldsymbol{{\epsilon}}$={eps}", fontsize=20, fontweight="bold")
        ax.set_xlabel(r"$\boldsymbol{N}$ (Sample Size)" if i==len(SF_list)-1 else "", fontsize=16, fontweight="bold")
        ax.set_ylabel(r"Obj. Diff. (%)" if j==0 else "", fontsize=16, fontweight="bold")
        _style_axes(ax); ax.set_ylim(_lo - _pad*0.3, gmax + _pad)
        ax.yaxis.set_major_locator(ticker.MaxNLocator(6))
        pmin, pmax = sub.diff_pct.min(), sub.diff_pct.max()
        zpad = max((pmax - pmin)*0.35, 0.006)
        mid = (ax.get_ylim()[0] + ax.get_ylim()[1]) / 2
        y0 = 0.08 if ((pmin+pmax)/2) > mid else 0.55
        axins = ax.inset_axes([0.42, y0, 0.55, 0.40])
        _dg(axins, sub, zoom=True)
        axins.set_ylim(pmin - zpad, pmax + zpad)
        axins.set_xlabel(""); axins.set_ylabel(""); axins.tick_params(labelsize=11)
        axins.yaxis.set_major_locator(ticker.MaxNLocator(4))
        axins.grid(True, linestyle="--", linewidth=0.9, alpha=0.5)
        for sp in axins.spines.values(): sp.set_edgecolor("#888888")
        ax.indicate_inset_zoom(axins, edgecolor="#888888", alpha=0.7, linewidth=1.0)
fig.tight_layout()
for ext in ("pdf","png"):
    fig.savefig(os.path.join(FIGDIR, f"optimality_gap_shared.{ext}"), bbox_inches="tight")
_rc.__exit__(None, None, None)
plt.show()
print(f"gap mean={gmean:.3f}% range=[{gmin:.3f}, {gmax:.3f}]% all_positive={bool((piv.diff_pct>0).all())}")

## 8. Peak memory - selected visualization

In [ ]:
import matplotlib.colors as mcolors
GB = 1/1024.0
NLIST = [50, 80, 100, 150, 200, 250, 300]
METHODS = [m for m in ["FICA","EIFICA-1shot","EIFICA-iter"] if m in set(res.m.unique())]

def mem_heatmap(net, fname=None):
    g = res[res.network == net]
    modes = [("EIFICA-1shot","EIFICA (direct)"), ("EIFICA-iter","EIFICA (iterative)")]
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.3)); fig.subplots_adjust(wspace=0.06)
    Ms = []
    for mk,_ in modes:
        M = np.full((len(EPS_THETA), len(NLIST)), np.nan)
        for i,(eps,th) in enumerate(EPS_THETA):
            sub = g[(g.eps==eps)&(g.theta==th)]
            fica = sub[sub.m=="FICA"].groupby("N").Mem.median()
            eif  = sub[sub.m==mk].groupby("N").Mem.median()
            for j,N in enumerate(NLIST):
                if N in fica.index and N in eif.index and eif.get(N,0)>0:
                    M[i,j] = fica[N]/eif[N]
        Ms.append(M)
    vmax = np.nanmax([np.nanmax(M) for M in Ms]); norm = mcolors.LogNorm(vmin=1.0, vmax=vmax)
    im = None
    for ci,(ax,(mk,ml),M) in enumerate(zip(axs, modes, Ms)):
        im = ax.imshow(M, aspect="auto", cmap="Greens", norm=norm)
        ax.set_xticks(range(len(NLIST))); ax.set_xticklabels(NLIST)
        ax.set_yticks(range(len(EPS_THETA)))
        ax.set_yticklabels([rf"$\epsilon$={e}, $\theta$={t}" for e,t in EPS_THETA] if ci==0 else [])
        ax.set_title(ml, fontsize=17, fontweight="bold", loc="left", color="#222")
        ax.set_xlabel(r"$\boldsymbol{N}$ (Training Scenarios)", fontsize=14, fontweight="bold")
        for lb in ax.get_xticklabels()+ax.get_yticklabels(): lb.set_fontweight("bold")
        ax.tick_params(labelsize=13, length=0)
        thr = norm(vmax)*0.55
        for i in range(len(EPS_THETA)):
            for j in range(len(NLIST)):
                if not np.isnan(M[i,j]):
                    ax.text(j, i, f"{M[i,j]:.0f}×", ha="center", va="center", fontsize=12,
                            fontweight="bold", color="white" if norm(M[i,j])>thr else "#1a3d1a")
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_xticks(np.arange(-.5,len(NLIST),1), minor=True)
        ax.set_yticks(np.arange(-.5,len(EPS_THETA),1), minor=True)
        ax.grid(which="minor", color="white", linewidth=2.5); ax.tick_params(which="minor", length=0)
    cbar = fig.colorbar(im, ax=axs, fraction=0.035, pad=0.02)
    cbar.set_label("Peak-memory reduction  (FICA / EIFICA)", fontsize=13, fontweight="bold")
    fig.suptitle(f"Peak-memory reduction of EIFICA over FICA  —  {NETLAB[net]}",
                 fontsize=15, fontweight="bold", y=1.02)
    if fname:
        for ext in ("pdf","png"): fig.savefig(os.path.join(FIGDIR, f"{fname}.{ext}"), bbox_inches="tight")
    plt.show()

mem_heatmap("case24_ieee_rts", "mem_heatmap_c24")
mem_heatmap("case118", "mem_heatmap_c118")

In [ ]:
GB = 1/1024.0
NLIST = [50, 80, 100, 150, 200, 250, 300]
METHODS = [m for m in ["FICA","EIFICA-1shot","EIFICA-iter"] if m in set(res.m.unique())]

def mem_bars(net, fname=None, Ns=(100,200,300)):
    g = res[res.network==net]
    fig, axs = plt.subplots(2,2, figsize=(10,7.1)); axs=axs.flatten()
    x = np.arange(len(Ns)); w = 0.26; ylo = 0.5
    allv = [(g[(g.eps==e)&(g.theta==t)&(g.m==mk)&(g.N==N)].Mem.median()*GB)
            for e,t in EPS_THETA for mk in METHODS for N in Ns]
    ytop = np.nanmax(allv)*2.6
    for i,(eps,th) in enumerate(EPS_THETA):
        ax=axs[i]; sub=g[(g.eps==eps)&(g.theta==th)]
        for k,mk in enumerate(METHODS):
            vals=[(sub[(sub.m==mk)&(sub.N==N)].Mem.median()*GB) for N in Ns]
            ax.bar(x+(k-1)*w, np.array(vals)-ylo, w, bottom=ylo, color=MCLR[mk],
                   edgecolor="white", linewidth=0.9, zorder=3, label=LABELS[mk])
            for xi,v in zip(x+(k-1)*w, vals):
                if v==v:
                    ax.text(xi, v*1.06, f"{v:.0f}" if v>=10 else f"{v:.1f}", ha="center",
                            va="bottom", fontsize=15, fontweight="bold", color=ACCENT, fontname="STIXGeneral")
        ax.set_yscale("log"); ax.set_ylim(ylo, ytop)
        ax.set_xticks(x); ax.set_xticklabels([f"N={n}" for n in Ns])
        ax.set_title(rf"{PANEL[i]}  $\boldsymbol{{\epsilon}}$={eps}, $\boldsymbol{{\theta}}$={th}",
                     fontsize=18, fontweight="bold", loc="left", color="#222")
        ax.set_ylabel("Peak Memory (GB)" if i%2==0 else "", fontsize=15, fontweight="bold")
        _style_axes(ax)
    mh=[Line2D([0],[0], color=MCLR[k], lw=8) for k in METHODS]
    fig.tight_layout(rect=(0,0,1,0.93))
    _p0=axs[0].get_position(); _p1=axs[1].get_position()
    fig.legend(mh,[LABELS[k] for k in METHODS],
               bbox_to_anchor=(_p0.x0,0.94,_p1.x1-_p0.x0,0.05), loc="lower left",
               ncol=len(METHODS), mode="expand", borderaxespad=0.0,
               prop={"weight":"bold","size":15}, frameon=True, framealpha=0.95,
               edgecolor="#cccccc", handletextpad=0.5)
    if fname:
        for ext in ("pdf","png"): fig.savefig(os.path.join(FIGDIR, f"{fname}.{ext}"), bbox_inches="tight")
    plt.show()

mem_bars("case118", "mem_bars_ieee118")

In [ ]:
GB = 1/1024.0
NLIST = [50, 80, 100, 150, 200, 250, 300]
METHODS = [m for m in ["FICA","EIFICA-1shot","EIFICA-iter"] if m in set(res.m.unique())]

def _msize(N):
    return 45 + (np.asarray(N, float)-50)/250.0*245

def mem_pareto(net, fname=None):
    g = res[res.network==net]
    fig, axs = plt.subplots(2,2, figsize=(10,7.4), sharex=True, sharey=True); axs=axs.flatten()
    for i,(eps,th) in enumerate(EPS_THETA):
        ax=axs[i]; sub=g[(g.eps==eps)&(g.theta==th)]
        tf=sub[sub.m=="FICA"].groupby("N").Time.median().reindex(NLIST)
        mf=(sub[sub.m=="FICA"].groupby("N").Mem.median().reindex(NLIST))*GB
        ax.plot(tf, mf, color=MCLR["FICA"], lw=1.4, alpha=0.5, zorder=2)
        for mk in METHODS:
            tmed=sub[sub.m==mk].groupby("N").Time.median().reindex(NLIST)
            mmed=(sub[sub.m==mk].groupby("N").Mem.median().reindex(NLIST))*GB
            ax.scatter(tmed, mmed, s=_msize(NLIST), color=MCLR[mk], marker=MMK[mk],
                       edgecolor="white", linewidth=1.2, alpha=0.9, zorder=3)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(rf"{PANEL[i]}  $\boldsymbol{{\epsilon}}$={eps}, $\boldsymbol{{\theta}}$={th}",
                     fontsize=18, fontweight="bold", loc="left", color="#222")
        if i//2==1: ax.set_xlabel("Solve Time (s)", fontsize=15, fontweight="bold")
        if i%2==0:  ax.set_ylabel("Peak Memory (GB)", fontsize=15, fontweight="bold")
        _style_axes(ax)
    szN=[50,150,300]
    sh=[Line2D([0],[0], marker="o", color="none", markerfacecolor="#9a9a9a", markeredgecolor="#555",
               markeredgewidth=1.0, markersize=np.sqrt(_msize(n))) for n in szN]
    leg1=axs[0].legend(sh,[f"N={n}" for n in szN], title="scenarios", loc="upper left",
                       labelspacing=1.1, borderpad=0.8, handletextpad=1.0,
                       prop={"weight":"bold","size":11}, title_fontsize=11, framealpha=0.9, edgecolor="#ccc")
    leg1.get_title().set_fontweight("bold"); axs[0].add_artist(leg1)
    mh=[Line2D([0],[0], color=MCLR[k], marker=MMK[k], lw=0, markersize=11,
               markerfacecolor=MCLR[k], markeredgecolor="white", markeredgewidth=1.2) for k in METHODS]
    fig.tight_layout(rect=(0,0,1,0.93))
    _p0=axs[0].get_position(); _p1=axs[1].get_position()
    fig.legend(mh,[LABELS[k] for k in METHODS],
               bbox_to_anchor=(_p0.x0,0.94,_p1.x1-_p0.x0,0.05), loc="lower left",
               ncol=len(METHODS), mode="expand", borderaxespad=0.0,
               prop={"weight":"bold","size":15}, frameon=True, framealpha=0.95,
               edgecolor="#cccccc", handletextpad=0.5)
    if fname:
        for ext in ("pdf","png"): fig.savefig(os.path.join(FIGDIR, f"{fname}.{ext}"), bbox_inches="tight")
    plt.show()

mem_pareto("case24_ieee_rts", "mem_pareto_rts24")